In [1]:
import pandas as pd
import re
import json
import math
import os
from difflib import SequenceMatcher
from ollama import chat, ChatResponse
from tqdm import tqdm

In [2]:
# =========================
# Settings
# =========================
FILE_PATH = "./data/c3_m_test.xlsx"

# 定義三個模型變數
M1 = "huihui_ai/internlm3-abliterated"
M2 = "glm4_9b"
M3 = "qwen3:8b"

MODELS = [
    (M1, False), 
    (M2, False), 
    (M3, False)
]

DEEPSEEK_THINK_MODEL = "qwen3:8b"
DEEPSEEK_THINK = True

TEMPERATURE = 0.0

def get_out_filename(model_name):
    clean_name = model_name.replace("/", "_").replace(":", "_")
    return f"result_{clean_name}.xlsx"

OUT_MODEL_FILE = {
    M1: get_out_filename(M1),
    M2: get_out_filename(M2),
    M3: get_out_filename(M3)
}

OUT_ENSEMBLE_FILE = "result_ensemble.xlsx"
OUT_SUMMARY_XLSX = "accuracy_summary.xlsx"
OUT_SUMMARY_CSV = "accuracy_summary.csv"

In [3]:
# =========================
# Ollama call
# =========================
def query_ollama(prompt, model, CONTEXT_LEN=1536, think=False) -> str:
    try:
        resp: ChatResponse = chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": CONTEXT_LEN, "temperature": TEMPERATURE},
            think=think
        )
        return resp.message.content
    except Exception as e:
        print(f"[Ollama Error] model={model} err={e}")
        return ""

In [4]:
# =========================
# Utility & Parsing
# =========================
def to_int_1_4(x):
    if x is None or pd.isna(x): return None
    try:
        xi = int(float(x))
        return xi if xi in (1, 2, 3, 4) else None
    except:
        return None

def to_conf_0_1(x):
    try:
        xf = float(x)
        return max(0.0, min(1.0, xf)) if not math.isnan(xf) else 0.0
    except:
        return 0.0

def _strip_think(text: str) -> str:
    if not text: return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def parse_answer_and_confidence(raw_text: str, options: dict) -> tuple[int | None, float | None]:
    if not raw_text: return None, None
    text = _strip_think(raw_text)
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            return to_int_1_4(obj.get("answer")), to_conf_0_1(obj.get("confidence"))
        except: pass
    m1 = re.search(r"Answer:\s*([1-4])", text, re.IGNORECASE)
    m2 = re.search(r"Confidence:\s*([0-9]*\.?[0-9]+)", text, re.IGNORECASE)
    ans = int(m1.group(1)) if m1 else None
    conf = float(m2.group(1)) if m2 else None
    return ans, conf

def build_prompt(article: str, question: str, valid_options: dict) -> str:
    options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
    valid_numbers = ", ".join([str(k) for k in valid_options.keys()])
    return f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
1) Select the best option among ({valid_numbers}). If an option is missing/empty, do not select it.
2) Output ONLY a JSON object with:
   - "answer": an integer in [{valid_numbers}]
   - "confidence": a number in [0, 1] representing your probability the answer is correct.

Example:
{{"answer": 2, "confidence": 0.78}}
""".strip()

In [5]:
# =========================
# New Ensemble Core Logic
# =========================
def get_majority_res(preds: dict, best_model: str):
    counts = {}
    for a in preds.values():
        if a is not None: counts[a] = counts.get(a, 0) + 1
    if not counts: return preds[best_model], True
    max_v = max(counts.values())
    winners = [k for k, v in counts.items() if v == max_v]
    is_tied = len(winners) > 1
    return (preds[best_model] if is_tied else winners[0]), is_tied

def get_weighted_res(preds: dict, confs: dict, best_model: str):
    scores = {1:0.0, 2:0.0, 3:0.0, 4:0.0}
    for m, a in preds.items():
        if a in scores: scores[a] += to_conf_0_1(confs.get(m))
    max_s = max(scores.values())
    if max_s == 0: return preds[best_model], True
    winners = [k for k, v in scores.items() if abs(v - max_s) < 1e-12]
    is_tied = len(winners) > 1
    return (preds[best_model] if is_tied else winners[0]), is_tied

In [7]:
# =========================
# Execution Functions
# =========================
def run_single_model(df, model_name, think):
    out_file = OUT_MODEL_FILE[model_name]
    if os.path.exists(out_file):
        res_df = pd.read_excel(out_file)
        return out_file, int(res_df["Correct"].sum()), len(res_df), (res_df["Correct"].sum()/len(res_df)*100)
    
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Predicting[{model_name}]"):
        options = {k: row[f"Option{k}"] for k in range(1,5) if pd.notna(row[f"Option{k}"])}
        raw = query_ollama(build_prompt(row["Article"], row["Question"], options), model_name, 1536, think)
        pred, conf = parse_answer_and_confidence(raw, options)
        actual = to_int_1_4(row["Answer"])
        is_corr = (pred == actual) if pred else False
        results.append({"ID":row["ID"],"Actual":actual,"Predicted":pred,"Confidence":conf,"Correct":is_corr,"Model_Response":raw})
    
    out_df = pd.DataFrame(results)
    out_df.to_excel(out_file, index=False)
    return out_file, int(out_df["Correct"].sum()), len(out_df), (out_df["Correct"].sum()/len(out_df)*100)

In [8]:
def run_all_ensembles(df_base, outputs, best_model):
    data = {m: pd.read_excel(outputs[m]) for m in MODELS_NAMES}
    total = len(df_base)
    actuals = data[best_model]["Actual"].tolist()
    
    ensemble_df = pd.DataFrame({"ID": data[best_model]["ID"], "Actual": actuals})
    
    # 統計用容器
    stats = {}

    for mode in ["Maj", "Weight"]:
        is_weighted = (mode == "Weight")
        res_list, tied_list, think_final, think_res = [], [], [], []
        
        # 統計變數
        tied_count, corr_count = 0, 0
        t_none, t_not_none, t_none_corr, t_not_none_corr = 0, 0, 0, 0
        
        for i in tqdm(range(total), desc=f"Ensemble {mode}"):
            p_row = {m: to_int_1_4(data[m].iloc[i]["Predicted"]) for m in MODELS_NAMES}
            c_row = {m: data[m].iloc[i]["Confidence"] for m in MODELS_NAMES}
            
            # 基本投票
            if not is_weighted: ans, tied = get_majority_res(p_row, best_model)
            else: ans, tied = get_weighted_res(p_row, c_row, best_model)
            
            res_list.append(ans)
            tied_list.append(tied)
            if tied: tied_count += 1
            
            # Thinking Logic
            final_think_ans = ans
            if tied:
                row = df_base.iloc[i]
                opts = {k: row[f"Option{k}"] for k in range(1,5) if pd.notna(row[f"Option{k}"])}
                raw = query_ollama(build_prompt(row["Article"], row["Question"], opts), DEEPSEEK_THINK_MODEL, 3192, DEEPSEEK_THINK)
                ds_ans, _ = parse_answer_and_confidence(raw, opts)
                ds_ans = to_int_1_4(ds_ans)
                think_res.append(raw)
                
                if ds_ans is None:
                    t_none += 1
                    final_think_ans = p_row[best_model]
                    if final_think_ans == actuals[i]: t_none_corr += 1
                else:
                    t_not_none += 1
                    final_think_ans = ds_ans
                    if final_think_ans == actuals[i]: t_not_none_corr += 1
            else:
                think_res.append("")
                
            think_final.append(final_think_ans)
            if final_think_ans == actuals[i]: corr_count += 1

        # 存入 DataFrame
        ensemble_df[f"{mode}_Base"] = res_list
        ensemble_df[f"{mode}_Is_Tied"] = tied_list
        ensemble_df[f"{mode}_Think_Final"] = think_final
        
        # 計算基礎正確率 (不含 thinking)
        base_corr = sum(1 for j in range(total) if res_list[j] == actuals[j])
        stats[mode] = (base_corr, tied_count, corr_count, t_none, t_not_none, t_none_corr, t_not_none_corr)

    ensemble_df.to_excel(OUT_ENSEMBLE_FILE, index=False)
    return stats

In [9]:
# =========================
# Main Entry
# =========================
df = pd.read_excel(FILE_PATH)
MODELS_NAMES = [m[0] for m in MODELS]
outputs = {}
model_accs = {}

for m_name, think in MODELS:
    out_f, corr, tot, acc = run_single_model(df, m_name, think)
    outputs[m_name] = out_f
    model_accs[m_name] = acc

best_model = max(model_accs, key=model_accs.get)
all_stats = run_all_ensembles(df, outputs, best_model)

# 打印最終報告
print("\n" + "="*50)
print("FINAL ENSEMBLE REPORT")
print("="*50)
total = len(df)

for mode, s in all_stats.items():
    base_corr, tied, think_corr, t_none, t_not_none, t_none_c, t_not_none_c = s
    print(f"\n[{mode} Vote Strategy]")
    print(f"1. 唯一多數筆數: {total-tied} ({(total-tied)/total*100:.2f}%)")
    print(f"2. 無唯一多數(平票): {tied} ({tied/total*100:.2f}%) -> 採用 Thinking")
    print(f"3. 基礎正確率 (不含 Thinking): {base_corr/total*100:.2f}%")
    print(f"4. Thinking 詳細統計:")
    if tied > 0:
        print(f"   - Thinking 回答 None (依賴 Best Model): {t_none} 筆 ({t_none/tied*100:.2f}%)")
        print(f"   - Thinking 成功回答筆數: {t_not_none} 筆 ({t_not_none/tied*100:.2f}%)")
        print(f"   - 最終正確率 (含 Thinking): {think_corr/total*100:.2f}%")
        print(f"   - Thinking 獨立正確率 (不含 None): {(t_not_none_c/t_not_none*100 if t_not_none>0 else 0):.2f}%")
        print(f"   - Best Model 救援正確率 (當 Thinking 為 None): {(t_none_c/t_none*100 if t_none>0 else 0):.2f}%")

print("\n[Done] 所有結果已儲存至 result_ensemble.xlsx")

Ensemble Weight: 100%|█████████████████████████████████████████████████████████████| 2002/2002 [04:10<00:00,  8.00it/s]



FINAL ENSEMBLE REPORT

[Maj Vote Strategy]
1. 唯一多數筆數: 1994 (99.60%)
2. 無唯一多數(平票): 8 (0.40%) -> 採用 Thinking
3. 基礎正確率 (不含 Thinking): 94.56%
4. Thinking 詳細統計:
   - Thinking 回答 None (依賴 Best Model): 2 筆 (25.00%)
   - Thinking 成功回答筆數: 6 筆 (75.00%)
   - 最終正確率 (含 Thinking): 94.71%
   - Thinking 獨立正確率 (不含 None): 83.33%
   - Best Model 救援正確率 (當 Thinking 為 None): 50.00%

[Weight Vote Strategy]
1. 唯一多數筆數: 1996 (99.70%)
2. 無唯一多數(平票): 6 (0.30%) -> 採用 Thinking
3. 基礎正確率 (不含 Thinking): 94.56%
4. Thinking 詳細統計:
   - Thinking 回答 None (依賴 Best Model): 1 筆 (16.67%)
   - Thinking 成功回答筆數: 5 筆 (83.33%)
   - 最終正確率 (含 Thinking): 94.66%
   - Thinking 獨立正確率 (不含 None): 80.00%
   - Best Model 救援正確率 (當 Thinking 為 None): 0.00%

[Done] 所有結果已儲存至 result_ensemble.xlsx


In [3]:
import pandas as pd
import re
import json
import math
import os
from difflib import SequenceMatcher
from ollama import chat, ChatResponse
from tqdm import tqdm

# =========================
# Settings
# =========================
FILE_PATH = "./data/c3_m_test.xlsx"

M1 = "huihui_ai/internlm3-abliterated"
M2 = "glm4_9b"
M3 = "qwen3:8b"

MODELS = [(M1, False), (M2, False), (M3, False)]
MODELS_NAMES = [M1, M2, M3]

DEEPSEEK_THINK_MODEL = "deepseek-r1:8b"
DEEPSEEK_THINK = True

TEMPERATURE = 0.0

def get_out_filename(model_name):
    clean_name = model_name.replace("/", "_").replace(":", "_")
    return f"result_{clean_name}.xlsx"

OUT_MODEL_FILE = {m: get_out_filename(m) for m in MODELS_NAMES}
OUT_ENSEMBLE_FILE = "result_ensemble.xlsx"

# =========================
# Ollama call & Parsing
# =========================
def query_ollama(prompt, model, CONTEXT_LEN = 1536, think=False) -> str:
    try:
        resp: ChatResponse = chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": CONTEXT_LEN, "temperature": TEMPERATURE},
            think=think
        )
        return resp.message.content
    except Exception as e:
        print(f"[Ollama Error] model={model} err={e}")
        return ""

def to_int_1_4(x):
    if x is None or pd.isna(x): return None
    try:
        xi = int(float(x))
        return xi if xi in (1, 2, 3, 4) else None
    except: return None

def to_conf_0_1(x):
    try:
        xf = float(x)
        return max(0.0, min(1.0, xf)) if not math.isnan(xf) else 0.0
    except: return 0.0

def _strip_think(text: str) -> str:
    if not text: return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def parse_answer_and_confidence(raw_text: str, options: dict) -> tuple[int | None, float | None]:
    if not raw_text: return None, None
    text = _strip_think(raw_text)
    # 嘗試 JSON 提取
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            return to_int_1_4(obj.get("answer")), to_conf_0_1(obj.get("confidence"))
        except: pass
    # 嘗試 正則 提取
    m1 = re.search(r"Answer:\s*([1-4])", text, re.IGNORECASE)
    ans = int(m1.group(1)) if m1 else None
    return ans, 0.5 # 默認置信度

def build_prompt(article: str, question: str, valid_options: dict) -> str:
    options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
    valid_numbers = ", ".join([str(k) for k in valid_options.keys()])
    return f"Article:\n{article}\n\nQuestion:\n{question}\n\nOptions:\n{options_text}\n\nInstruction: Output ONLY a JSON object with 'answer' (int) and 'confidence' (float)."

# =========================
# Voting Logic
# =========================
def get_majority_res(preds: dict, best_model: str):
    counts = {}
    for a in preds.values():
        if a is not None: counts[a] = counts.get(a, 0) + 1
    if not counts: return preds[best_model], True
    max_v = max(counts.values())
    winners = [k for k, v in counts.items() if v == max_v]
    is_tied = len(winners) > 1
    return (preds[best_model] if is_tied else winners[0]), is_tied

def get_weighted_res(preds: dict, confs: dict, best_model: str):
    scores = {1:0.0, 2:0.0, 3:0.0, 4:0.0}
    for m, a in preds.items():
        if a in scores: scores[a] += to_conf_0_1(confs.get(m))
    max_s = max(scores.values())
    if max_s == 0: return preds[best_model], True
    winners = [k for k, v in scores.items() if abs(v - max_s) < 1e-12]
    is_tied = len(winners) > 1
    return (preds[best_model] if is_tied else winners[0]), is_tied

# =========================
# Core Execution
# =========================
def run_single_model(df, model_name, think):
    out_file = OUT_MODEL_FILE[model_name]
    if os.path.exists(out_file):
        res_df = pd.read_excel(out_file)
        return out_file, int(res_df["Correct"].sum()), len(res_df), (res_df["Correct"].sum()/len(res_df)*100)
    
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Predicting[{model_name}]"):
        opts = {k: row[f"Option{k}"] for k in range(1,5) if pd.notna(row[f"Option{k}"])}
        raw = query_ollama(build_prompt(row["Article"], row["Question"], opts), model_name, 1536, think)
        pred, conf = parse_answer_and_confidence(raw, opts)
        actual = to_int_1_4(row["Answer"])
        results.append({"ID":row["ID"],"Actual":actual,"Predicted":pred,"Confidence":conf,"Correct":(pred==actual),"Model_Response":raw})
    
    out_df = pd.DataFrame(results)
    out_df.to_excel(out_file, index=False)
    return out_file, int(out_df["Correct"].sum()), len(out_df), (out_df["Correct"].sum()/len(out_df)*100)

def run_all_ensembles(df_base, outputs, best_model):
    data = {m: pd.read_excel(outputs[m]) for m in MODELS_NAMES}
    total = len(df_base)
    actuals = data[best_model]["Actual"].tolist()
    ensemble_df = pd.DataFrame({"ID": data[best_model]["ID"], "Actual": actuals})
    
    final_stats = {}

    for mode in ["Maj", "Weight"]:
        is_weighted = (mode == "Weight")
        res_list, tied_list, think_final, think_raw_list = [], [], [], []
        
        # 統計
        tied_count, corr_count = 0, 0
        t_none, t_not_none, t_none_corr, t_not_none_corr = 0, 0, 0, 0
        
        for i in tqdm(range(total), desc=f"Processing {mode} + Thinking"):
            p_row = {m: to_int_1_4(data[m].iloc[i]["Predicted"]) for m in MODELS_NAMES}
            c_row = {m: data[m].iloc[i]["Confidence"] for m in MODELS_NAMES}
            
            # 1. 基礎投票
            if not is_weighted: ans, tied = get_majority_res(p_row, best_model)
            else: ans, tied = get_weighted_res(p_row, c_row, best_model)
            
            res_list.append(ans)
            tied_list.append(tied)
            
            # 2. Thinking 邏輯 (僅在平票時觸發)
            final_ans = ans
            raw_output = ""
            if tied:
                tied_count += 1
                row = df_base.iloc[i]
                opts = {k: row[f"Option{k}"] for k in range(1,5) if pd.notna(row[f"Option{k}"])}
                raw_output = query_ollama(build_prompt(row["Article"], row["Question"], opts), DEEPSEEK_THINK_MODEL, 8192, DEEPSEEK_THINK)
                ds_ans, _ = parse_answer_and_confidence(raw_output, opts)
                ds_ans = to_int_1_4(ds_ans)
                
                if ds_ans is None:
                    t_none += 1
                    final_ans = p_row[best_model] # 採用正確率最高的LLM
                    if final_ans == actuals[i]: t_none_corr += 1
                else:
                    t_not_none += 1
                    final_ans = ds_ans
                    if final_ans == actuals[i]: t_not_none_corr += 1
            
            think_final.append(final_ans)
            think_raw_list.append(raw_output)
            if final_ans == actuals[i]: corr_count += 1

        ensemble_df[f"{mode}_Base"] = res_list
        ensemble_df[f"{mode}_Is_Tied"] = tied_list
        ensemble_df[f"{mode}_Think_Final"] = think_final
        ensemble_df[f"{mode}_Think_Raw"] = think_raw_list # 這是新增的欄位，記錄 None 的原因
        
        base_corr = sum(1 for j in range(total) if res_list[j] == actuals[j])
        final_stats[mode] = (base_corr, tied_count, corr_count, t_none, t_not_none, t_none_corr, t_not_none_corr)

    ensemble_df.to_excel(OUT_ENSEMBLE_FILE, index=False)
    return final_stats

# =========================
# Entry
# =========================
df = pd.read_excel(FILE_PATH)
outputs = {}
model_accs = {}

for m_name, think in MODELS:
    out_f, corr, tot, acc = run_single_model(df, m_name, think)
    outputs[m_name], model_accs[m_name] = out_f, acc

best_model = max(model_accs, key=model_accs.get)
results = run_all_ensembles(df, outputs, best_model)

# 打印報告
print("\n" + "="*40 + "\nSTATISTICS REPORT\n" + "="*40)
for mode, s in results.items():
    b_c, tied, t_c, t_n, t_nn, t_n_c, t_nn_c = s
    total = len(df)
    print(f"\n[{mode} Vote + Thinking]")
    print(f"- 唯一多數筆數: {total-tied} ({(total-tied)/total*100:.2f}%)")
    print(f"- 無唯一多數筆數: {tied} ({tied/total*100:.2f}%)")
    print(f"- Thinking 回答 None 比例: {t_n}/{tied} ({(t_n/tied*100 if tied>0 else 0):.2f}%)")
    print(f"- Thinking 成功回答正確率 (不含None): {(t_nn_c/t_nn*100 if t_nn>0 else 0):.2f}%")
    print(f"- Best Model 救援正確率 (當None時): {(t_n_c/t_n*100 if t_n>0 else 0):.2f}%")
    print(f"- 策略最終總準確率: {t_c/total*100:.2f}%")

print(f"\n[完成] 詳細回答已記錄於 {OUT_ENSEMBLE_FILE} 的 '_Think_Raw' 欄位中。")

Processing Weight + Thinking: 100%|████████████████████████████████████████████████| 2002/2002 [01:25<00:00, 23.50it/s]



STATISTICS REPORT

[Maj Vote + Thinking]
- 唯一多數筆數: 1994 (99.60%)
- 無唯一多數筆數: 8 (0.40%)
- Thinking 回答 None 比例: 2/8 (25.00%)
- Thinking 成功回答正確率 (不含None): 50.00%
- Best Model 救援正確率 (當None時): 0.00%
- 策略最終總準確率: 94.56%

[Weight Vote + Thinking]
- 唯一多數筆數: 1996 (99.70%)
- 無唯一多數筆數: 6 (0.30%)
- Thinking 回答 None 比例: 1/6 (16.67%)
- Thinking 成功回答正確率 (不含None): 40.00%
- Best Model 救援正確率 (當None時): 0.00%
- 策略最終總準確率: 94.56%

[完成] 詳細回答已記錄於 result_ensemble.xlsx 的 '_Think_Raw' 欄位中。
